In [42]:
import os
import pandas as pd
import numpy as np
import subprocess

In [43]:
MICROARRAY_BC_SCRIPT = "Rscript ../rscripts/microarray_bc.r"
RNA_SEQ_BC_SCRIPT = "Rscript ../rscripts/rnaseq_bc.r"

In [44]:
coldata_cols = ["sample_name", "condition", "batch"]

In [45]:
user_dir = "../app/example_data/microarray_session/"

In [46]:
data_type = "microarray"

In [47]:
reference_level = "normal"

In [48]:
contrast_level = "endometriosis"

In [49]:
counts_path = f"{user_dir}counts.tsv"
coldata_path = f"{user_dir}coldata.tsv"

counts_exists = os.path.isfile(counts_path)
coldata_exists = os.path.isfile(coldata_path)

In [50]:
if not counts_exists and not coldata_exists:
    print("Error: Counts and coldata files not found")
elif not counts_exists:
    print("Error: Counts file not found")
elif not coldata_exists:
    print("Error: Coldata file not found")

In [72]:
counts_cols = list(pd.read_csv(counts_path, nrows =1, sep="\t"))
counts = pd.read_csv(counts_path, usecols = \
        [i for i in counts_cols if i.lower() != "entrez_gene_id"], sep='\t')

coldata = pd.read_csv(coldata_path, usecols = coldata_cols, sep='\t')

In [73]:
# TODO: remove samples with levels other than the reference/contrast levels
factor_levels = [reference_level, contrast_level]
filtered_samples = coldata.loc[coldata['condition'].isin(factor_levels)]

In [80]:
samples_to_keep

,sample_name,condition,batch
0,GSM109815,normal,1
2,GSM109816,normal,1
3,GSM109817,normal,1
4,GSM109828,normal,1
5,GSM109829,normal,1
...,...,...,...
149,GSM1256796,normal,4
150,GSM1256797,normal,4
151,GSM1256798,normal,4
152,GSM1256799,normal,4


In [74]:
sample_names = coldata.loc[:, "sample_name"]

In [75]:
counts

,symbol,GSM109815,GSM229815,GSM109816,GSM109817,GSM109828,GSM109829,GSM109830,GSM109831,GSM150190,...,GSM1256786,GSM1256787,GSM1256788,GSM1256791,GSM1256793,GSM1256796,GSM1256797,GSM1256798,GSM1256799,GSM1256800
0,A1BG,5.685566,5.685566,6.309144,6.558803,5.676761,5.713494,5.651049,6.095096,6.734897,...,5.807815,5.799219,6.019293,5.726396,5.767185,5.945817,5.736468,5.966595,5.868001,5.931074
1,A1BG-AS1,6.354643,6.354643,7.025457,7.019929,6.820971,6.986295,7.019873,6.614798,7.162332,...,6.700733,6.763380,6.717756,6.878098,6.691541,6.742213,6.825399,6.669743,6.579118,6.651996
2,A1CF,6.017202,6.017202,6.806242,5.876322,6.075240,6.048896,6.123009,6.480756,6.343271,...,6.245288,6.031817,5.796515,6.262572,6.067918,6.291459,5.939247,6.274130,6.093899,5.928299
3,A2M,10.705147,10.705147,10.982733,11.381476,11.434625,11.319421,11.669190,11.679939,11.021458,...,11.147112,11.202096,11.563820,11.578159,11.268654,11.595849,11.414081,11.245570,11.330142,11.243454
4,A2M-AS1,6.193356,6.193356,6.496668,6.377511,6.072465,5.664001,6.158695,5.950787,6.176829,...,6.171834,6.317519,6.506197,5.722692,6.196140,6.186685,6.062160,6.451924,6.235170,5.969264
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21402,ZYG11A,5.896223,5.896223,5.963051,5.816302,5.816189,6.331827,5.990648,5.990648,5.985147,...,5.937042,5.867404,5.882119,5.946743,5.960328,5.966710,5.984837,6.071701,6.089232,5.922221
21403,ZYG11B,7.953928,7.953928,7.819311,8.357682,8.419856,7.860623,7.962587,8.031315,8.291714,...,8.336214,8.311109,8.311155,8.055209,8.331800,8.282842,8.096207,8.371499,8.309326,8.257878
21404,ZYX,10.110586,10.110586,10.359008,10.440895,9.986121,10.122976,10.791009,10.945596,11.130939,...,10.244438,10.337396,10.643108,10.509105,10.337931,10.408817,10.591364,10.329832,10.363148,10.364964
21405,ZZEF1,7.695005,7.695005,7.715166,7.836197,8.193168,8.122933,8.089190,8.007628,7.800383,...,7.841115,8.161412,7.970211,7.936386,7.847303,7.833776,8.204047,7.786818,7.805458,7.928791


In [76]:
all_indices = set(range(0,coldata.shape[0]))
sample_indices_to_keep = set(samples_to_keep.index.values)
sample_indices_to_drop = np.asarray(list(all_indices - sample_indices_to_keep))
sample_indices_to_drop += 1

array([2])

In [77]:
counts.drop(counts.columns[sample_indices_to_drop], axis=1, inplace=True)

,symbol,GSM109815,GSM109816,GSM109817,GSM109828,GSM109829,GSM109830,GSM109831,GSM150190,GSM150191,...,GSM1256786,GSM1256787,GSM1256788,GSM1256791,GSM1256793,GSM1256796,GSM1256797,GSM1256798,GSM1256799,GSM1256800
0,A1BG,5.685566,6.309144,6.558803,5.676761,5.713494,5.651049,6.095096,6.734897,6.311050,...,5.807815,5.799219,6.019293,5.726396,5.767185,5.945817,5.736468,5.966595,5.868001,5.931074
1,A1BG-AS1,6.354643,7.025457,7.019929,6.820971,6.986295,7.019873,6.614798,7.162332,7.692469,...,6.700733,6.763380,6.717756,6.878098,6.691541,6.742213,6.825399,6.669743,6.579118,6.651996
2,A1CF,6.017202,6.806242,5.876322,6.075240,6.048896,6.123009,6.480756,6.343271,6.492033,...,6.245288,6.031817,5.796515,6.262572,6.067918,6.291459,5.939247,6.274130,6.093899,5.928299
3,A2M,10.705147,10.982733,11.381476,11.434625,11.319421,11.669190,11.679939,11.021458,10.674658,...,11.147112,11.202096,11.563820,11.578159,11.268654,11.595849,11.414081,11.245570,11.330142,11.243454
4,A2M-AS1,6.193356,6.496668,6.377511,6.072465,5.664001,6.158695,5.950787,6.176829,5.792399,...,6.171834,6.317519,6.506197,5.722692,6.196140,6.186685,6.062160,6.451924,6.235170,5.969264
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21402,ZYG11A,5.896223,5.963051,5.816302,5.816189,6.331827,5.990648,5.990648,5.985147,6.172638,...,5.937042,5.867404,5.882119,5.946743,5.960328,5.966710,5.984837,6.071701,6.089232,5.922221
21403,ZYG11B,7.953928,7.819311,8.357682,8.419856,7.860623,7.962587,8.031315,8.291714,7.783332,...,8.336214,8.311109,8.311155,8.055209,8.331800,8.282842,8.096207,8.371499,8.309326,8.257878
21404,ZYX,10.110586,10.359008,10.440895,9.986121,10.122976,10.791009,10.945596,11.130939,10.951452,...,10.244438,10.337396,10.643108,10.509105,10.337931,10.408817,10.591364,10.329832,10.363148,10.364964
21405,ZZEF1,7.695005,7.715166,7.836197,8.193168,8.122933,8.089190,8.007628,7.800383,7.488816,...,7.841115,8.161412,7.970211,7.936386,7.847303,7.833776,8.204047,7.786818,7.805458,7.928791


In [ ]:
counts.to_csv(f"{user_dir}counts_bc-in.tsv", sep='\t', index=False)
coldata.to_csv(f"{user_dir}coldata_bc-in.tsv", sep='\t', index=False)

In [ ]:
if data_type == "microarray":
    subprocess.Popen([f"{MICROARRAY_BC_SCRIPT} {user_dir}counts_bc-in.tsv "\
                        f"{user_dir}coldata_bc-in.tsv {user_dir}"], shell=True)
elif data_type == "RNA-Seq":
    subprocess.Popen([f"{RNA_SEQ_BC_SCRIPT} {user_dir}counts_bc-in.tsv "\
                        f"{user_dir}coldata_bc-in.tsv {user_dir}"], shell=True)